I2 portable reader candidate. New outputs are isolated under runs/. Constructed examples demonstrate methods; fixed filing facts/figures are historical teaching inputs. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch12_statements_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第 12 章：财务报表与量化研究

本 Notebook 调用 **src/quant_course/statement_metrics.py** 中的类型化函数，复算 Apple FY2024 与 FY2023 的已审计报表行，并对照 Spire FY2023 原始申报与重述申报。

Apple 与 Spire 的报表金额来自正文链接的 SEC filings。PIT 时间戳、issuer 标签与市值示例是构造数据；本 Notebook 不读取真实历史股价、供应商 ingestion 日志，也不计算因子绩效。所有美元金额必须先匹配 currency、unit、period 和 filing accession。

In [ ]:
from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from quant_course.statement_metrics import (
    apple_facts,
    derive_accounting_factors,
    fcf_from_statement_lines,
    fcf_yield,
    point_in_time_revision_demo,
    revenue_growth,
    spire_cash_flow_facts,
)

print("工作目录：", ROOT)
print("单位约定：Apple USD millions；Spire USD thousands")


## Apple：利润率、现金流与账面杠杆

FY2024 与 FY2023 数字均取自同一个 2024 Form 10-K accession 的并列表。资本开支在原表是现金流出，本模块把它转成正支出后计算 FCF。净债定义只包括 commercial paper 与 term debt，减 cash and cash equivalents；不扣 marketable securities，也不纳入 leases。

In [ ]:
apple_2024 = apple_facts(2024)
apple_2023 = apple_facts(2023)
metrics_2024 = derive_accounting_factors(apple_2024)
metrics_2023 = derive_accounting_factors(apple_2023)
revenue_growth_2024 = revenue_growth(
    apple_2024["revenue"],
    apple_2023["revenue"],
)

print({
    "FY2024_revenue_USD_millions": apple_2024["revenue"].value,
    "FY2024_FCF_USD_millions": metrics_2024["fcf"],
    "FY2024_FCF_margin": metrics_2024["fcf_margin"],
    "FY2024_operating_margin": metrics_2024["operating_margin"],
    "FY2024_EBITDA_proxy_USD_millions": metrics_2024["ebitda_proxy"],
    "FY2024_net_debt_USD_millions": metrics_2024["net_debt"],
    "FY2024_balance_sheet_residual": metrics_2024["balance_sheet_residual"],
    "FY2024_revenue_growth_vs_FY2023": revenue_growth_2024,
})
assert metrics_2024["fcf"] == 108_807.0
assert metrics_2023["fcf"] == 99_584.0
assert metrics_2024["balance_sheet_residual"] == 0.0
assert math.isclose(revenue_growth_2024, 391_035 / 383_285 - 1, abs_tol=1e-14)


## Spire：版本必须分开

两个版本对应不同 accession。CapEx cash-flow line 使用原报表现金流符号（负值）；先转为正支出，再计算 CFO − CapEx。相同的本章 FCF 数值不表示其他会计项目没有变化。

In [ ]:
spire_rows = []
for revision in ("original", "restated"):
    cfo, capex_cash_flow = spire_cash_flow_facts(revision)
    fcf = fcf_from_statement_lines(cfo, capex_cash_flow)
    spire_rows.append({
        "revision": revision,
        "accession": cfo.accession,
        "CFO_USD_thousands": cfo.value,
        "CapEx_spend_USD_thousands": -capex_cash_flow.value,
        "FCF_USD_thousands": fcf,
    })
spire_versions = pd.DataFrame(spire_rows)
display(spire_versions)
assert spire_versions["accession"].nunique() == 2
assert spire_versions["CFO_USD_thousands"].tolist() == [-23_622.0, -36_307.0]
assert spire_versions["CapEx_spend_USD_thousands"].tolist() == [30_037.0, 17_352.0]
assert spire_versions["FCF_USD_thousands"].tolist() == [-53_659.0, -53_659.0]


## Point-in-time 两时钟示例（构造数据）

这里的时间字符串明确写成 UTC，但整条时间线是构造数据，不是 Apple 或 Spire 的公告时间。筛选要求 available_at 与 ingested_at 均不晚于 signal_time。版本的新旧按来源 available_at 排序，ingested_at 只作资格门槛。

In [ ]:
pit = point_in_time_revision_demo()
display(pit[["signal_time", "accession", "revision_status", "cfo_usd_millions"]])
assert pd.isna(pit.loc[0, "cfo_usd_millions"])
assert pit.loc[1, "revision_status"] == "original"
assert pit.loc[2, "revision_status"] == "original"
assert pit.loc[3, "revision_status"] == "restated"


## 构造市值只作单位演示

财报不包含本例所需历史市场市值。以下 4,000,000 USD millions 是显式构造输入，不是 Apple 实际市值，也不是可交易的 FCF yield 估计。

In [ ]:
demo_yield = fcf_yield(
    metrics_2024["fcf"],
    4_000_000.0,
    currency="USD",
    statement_unit="USD millions",
    market_cap_currency="USD",
    market_cap_unit="USD millions",
    market_cap_label="constructed teaching value",
)
print(f"Constructed FCF yield: {demo_yield:.4%}")
assert math.isclose(demo_yield, 108_807 / 4_000_000, abs_tol=1e-14)


## 已生成图

图中金额来自链接的 SEC 申报；Spire 横轴表示 accession 版本。

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch12_apple_cashflow.png")))
display(Image(filename=str(ROOT / "figures" / "a3" / "ch12_spire_revision.png")))


## 阅读限制

Apple 新闻稿页面只核到发布日期；SEC header 中的 raw acceptance 字符串未在这里转换成 UTC。Spire 8-K 的 report date、filed date、10-K/A 日期也保持各自语义。财报历史版本与真实供应商入库记录、证券映射、交易日历和市场价格尚未组成完整 PIT 面板，所以这些报表算术不能支持真实 FCF yield 历史或回测结论。